In [ ]:
# import numpy as np
# # 7 Design of the hydraulic components

# # 7.1.2 Hydraulic specification

# # 7.1.2.1 Best efficient point (BEP)
# mdot_opt = 0.3  # kg/s, optimal mass flow rate
# rho_0 = 1000.0  # kg/m^3
# H_opt = 300.0   # m, head at BEP
# n = 30000 # rpm
# Q_opt = mdot_opt / rho_0 # m^3/s, optimal flow rate

# # 7.1.2.2
# Q_max = 1.1 * Q_opt # m^3/s, maximum flow rate

# # 7.1.2.3
# # NPSH_A = ??

# # 7.1.3, channel model

# #STEP 1
# n_q = n * np.sqrt(Q_opt) / np.pow(H_opt, 0.75)  # Specific speed
# A_v = (42 + 53) * 0.5 / n_q # Area ratio 7.1.3.1

# #STEP 2 
# head_coeff_th = np.pow(A_v, 0.23) # Theoretical head coefficient
# c3q_d_u2 = 0.31 * np.pow(A_v, 0.45) # c3q/u2, Average velocity in diffuser throat / Radial speed at impeller outlet

# #STEP 3
# #Estimate theoretical hydraulic efficiency with T3.9, radial pump single stage

# Q_ref = 1 # T.(3.9)
# m = 0.1 * 1 * np.pow(Q_ref / Q_opt, 0.15) * np.pow(45 / n_q, 0.06) # Efficiency Exponent Eq. (3.9.1)
# # eff_opt = 1 - 0.095 * np.pow(Q_ref/ Q_opt, m) - 0.3 * np.pow(0.35 - np.log10(n_q / 23), 2) * np.pow(Q_ref / Q_opt, 0.05) # Overall efficiency of single stage radial pump Eq.(3.9.1), might not be applicable above Q>0.005, for 0.0005 gives negative so it can't be used.
# eff_opt = 0.4 #Table 2.3, lowest number

# eff_h_opt = 0.5 * (1 + eff_opt) # Approximation for small pumps and partlead Eq.(3.28a) Apprently this equation is good for even Q=0 and partload.

# m_h = 0.08 * np.pow(Q_ref / Q_opt, 0.08) * (1 - eff_opt)
# eff_h_opt2 = 1 - 0.055 * np.pow(Q_ref/ Q_opt, m_h) - 0.2 * np.pow(0.26 - np.log10(n_q / 25), 2) * np.pow(Q_ref / Q_opt, 0.1) # Hydraulic efficiency of single stage radial pumpss Eq.(3.9.1), might not be applicable above Q>0.005 but nothing else can be done
# eff_h_opt = min(eff_h_opt, eff_h_opt2)  # Take the minimum of the two estimates


# head_coeff_opt = eff_h_opt * head_coeff_th  # Head coefficient 7.1.3.3

# g = 9.81  # m/s^2, gravitational acceleration
# d_2 = 60 / (np.pi * n) * np.sqrt(2 * g * H_opt / head_coeff_opt) # Impeller outlet diameter Eq.(7.1.3)
# u_2 = np.pi * d_2 * n / 60  # Radial speed at r=d2/2
# c3q = u_2 * c3q_d_u2

# #STEP 4
# z_Le = 1 # Number of diffuser vanes or volute channels, doesn't really make sense for an annular volute, Le is volute, La is Laufrad or impelelr in english.
# A_3q = Q_opt / (z_Le * c3q) # Throat area of diffuser

# #STEP 5
# # Relative outlet width b2star
# lognq = np.log10(n_q)
# b2star = np.pow(10, -0.2 * lognq * lognq + 1.35 * lognq - 2.6) # Fig 7.2, quadratic regression from https://www.desmos.com/calculator/l44pulw06b, better than given 7.1 cubic fit

# z_La = 6 # Number of impeller blades
# A_2q = A_v * z_Le * A_3q / z_La # Area between vanes at impeller outlet 7.1.3.5
# b_2 = A_2q / d_2 # It says d_2a in fig 7.2 and 7.2.1.9, subscripts say a means outer streamline, but I don't know how to calculate that.

# #STEP 6
# a_2 = A_2q / b_2

# Sizing using 7.2.1 Determination of main dimensions
import numpy as np

# 7.2.1.1 Boundary conditions
mdot_opt = 0.3  # kg/s, optimal mass flow rate
rho_0 = 1000.0  # kg/m^3
H_opt = 200.0   # m, head at BEP
n = 20000 # rpm
Q_opt = mdot_opt / rho_0 # m^3/s, optimal flow rate

# 7.2.1.2 Efficiencies estimation.
eff_opt = 0.2 #Table 2.3, lowest number
eff_h_opt = 0.5 * (1 + eff_opt)

# 7.2.1.3 Shaft diameter
d_w = 0.01 # 1cm shaft

# 7.2.1.4 Impeller outer diameter d2
n_q = n * np.sqrt(Q_opt) / np.power(H_opt, 0.75)  # Specific speed
psi_coeff_opt = 1.21 * np.exp(-0.77 * n_q / 100) # Head coefficient Eq.(3.26), best fit from Fig 3.16.

d_2 = 60 / (np.pi * n) * np.sqrt(2 * 9.81 * H_opt / psi_coeff_opt)  # Impeller outlet diameter Eq.(7.1.3)

#7.2.1.5 Blade number
z_La = 6 # Just for now!! Many requirements play into this, I haven't read

#7.2.1.6 Impeller inlet diameter
#Sized with method A, minimum relative velocity at impeller inlet
d_n_star = 0.0 # Hub diameter
delta_r = 1 # Swirl number Eq.(T.7.1.4)
f_d_1 = 1.2 # Eq.(T.7.1.4)
d_1_star = f_d_1 * np.sqrt(d_n_star * d_n_star + 1.5e-3 * psi_coeff_opt * np.pow(n_q, 1.33) / np.pow(delta_r, 0.67))
d_1 = d_1_star * d_2

#7.2.1.7 Blade Inlet Diameter
#don't know for overhung impeller?
d_1_i = d_1 #for now...?

#7.2.1.8 Impeller blade inlet angles
#Assume 90 for Barske
beta_1B = np.deg2rad(90)

#Inlet velocity triangle T3.1
u_1 = np.pi * d_1 * n / 60
d_n = d_n_star * d_2  # Hub diameter
A_1 = np.pi / 4 * (d_1 ** 2 - d_n ** 2)  # Area at impeller inlet
c_1m = Q_opt / A_1  # Meridional velocity
c_1u = 0 # Assumed zero for now, no swirl at inlet
w_1 = np.sqrt(c_1m ** 2 + (u_1 - c_1u) ** 2)  # Relative velocity
phi_1 = c_1m / u_1 # Flow coefficient
beta_1 = np.arctan(c_1m / (u_1 - c_1u)) # Flow angle without blockage


#7.2.1.9 Outlet width b2
lognq = np.log10(n_q)
b_2star = np.pow(10, -0.2 * lognq * lognq + 1.35 * lognq - 2.6) # Fig 7.2, quadratic regression from https://www.desmos.com/calculator/l44pulw06b, better than given 7.1 cubic fit
b_2 = b_2star * d_2

#7.2.1.10 Outlet angle
beta_2B = np.deg2rad(90)

#Outlet velocity triangle T3.2
u_2 = np.pi * d_2 * n / 60 
c_2m = Q_opt / (np.pi * d_2 * b_2)
tau_2 = 1 # BLADE BLOCKAGE NOT DONE
phi_2 = c_2m / u_2
epsilon_lim = np.exp(-8.16 * np.sin(beta_2B) / z_La)
# Slip factor correction due to impeller inlet diameter
if epsilon_lim < d_1_star:
    k_w = 1
else:
    k_w = 1 - np.pow((d_1_star - epsilon_lim) / (1 - epsilon_lim), 3)
f_1 = 0.98 # For radial impellers
gamma = f_1 * (1 - np.sqrt(np.sin(beta_2B)) / z_La**0.7) * k_w # Slip factor
c_2u = u_2 * gamma # tan 90 = infinite

power = mdot_opt * 9.81 * H_opt / eff_opt

print(f"Boundary conditions:")
print(f"  {"Mass flow rate (mdot)":<30} {mdot_opt:<10.4g} kg/s")
print(f"  {"Outlet Head (H)":<30} {H_opt:<10.4g} m")
print(f"  {"Rotational speed (n)":<30} {n:<10.1g} rpm")
print(f"  {"Flow rate (Q)":<30} {Q_opt:<10.4g} m^3/s")
print(f"  {"Specific speed (n_q)":<30} {n_q:<10.4g}")
print(f"Outputs:")
print(f"  {"Efficiency":<30} {eff_opt:<10.4g}")
print(f"  {"Hydraulic efficiency":<30} {eff_h_opt:<10.4g}")
print(f"  {"Head coefficient":<30} {psi_coeff_opt:<10.4g}")
print(f"  {"Impeller outlet diameter (d2)":<30} {d_2*1000:<10.4g} mm")
print(f"  {"Impeller inlet diameter (d1)":<30} {d_1*1000:<10.4g} mm")
print(f"  {"Power required (P)":<30} {power:<10.4g} W")
print(f"  {'b2':<30} {b_2*1000:<10.4g} mm")


Boundary conditions:
  Mass flow rate (mdot)          0.3        kg/s
  Outlet Head (H)                200        m
  Rotational speed (n)           2e+04      rpm
  Flow rate (Q)                  0.0003     m^3/s
  Specific speed (n_q)           6.514     
Outputs:
  Efficiency                     0.2       
  Hydraulic efficiency           0.6       
  Head coefficient               1.151     
  Impeller outlet diameter (d2)  55.76      mm
  Impeller inlet diameter (d1)   9.666      mm
  Power required (P)             2943       W
  b2                             1.296      mm
